# Add Zones to Gold

## Import Packages

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType,
    BooleanType,
    DateType,
    TimestampType,
    DecimalType
)

In [0]:
from pyspark import pipelines as dp

## Variables

## Schema Definition

In [0]:
schema = StructType(
    [
        StructField(
            name="location_id",
            dataType=IntegerType(),
            nullable=False,
            metadata={"comment":"Shows the Location ID of each NYC Zone"}
        ),
        StructField(
            name="borough",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Borough Name"}
        ),
        StructField(
            name="zone",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Zone Name"}
        ),
        StructField(
            name="service_zone",
            dataType=StringType(),
            nullable=False,
            metadata={"comment":"Shows the Service Zone Name"}
        ),
        StructField(
            name="the_geom",
            dataType=StringType(),
            nullable=True,
            metadata={"comment": "Presents the Location as Polygon Data for a Shape Map"}
        ),
        StructField(
            name="shape_leng",
            dataType=DecimalType(30,20),
            nullable=True,
            metadata={"comment": "Shows the Length of the Shape"}
        ),
        StructField(
            name="shape_area",
            dataType=DecimalType(30,20),
            nullable=True,
            metadata={"comment": "Shows the Area of the Shape"}
        ),
        StructField(
            name="latitude",
            dataType=DecimalType(30,20),
            nullable=True,
            metadata={"comment": "Describes the center latitude of the zone"}
        ),
        StructField(
            name="longitude",
            dataType=DecimalType(30,20),
            nullable=True,
            metadata={"comment": "Describes the center longitude of the zone"}
        ),
    ]
)

## ETL

In [0]:
@dp.materialized_view(
    # Name der Zieltabelle
    name="analytics.gold.zones",
    # Beschreibung der Tabelle
    comment="This table shows the zones in NYC and their Polygon Data",
    # Liquid Clustering (Statt partitioning und Z-Order)
    cluster_by=[],
    cluster_by_auto=True,
    schema=schema,
)
def zones():
    df = spark.read.table("analytics.silver.geographics_stm_zone")

    df_exploded = df.withColumn("geo", F.from_json("the_geom", "type STRING, coordinates ARRAY<ARRAY<ARRAY<ARRAY<DOUBLE>>>>"))
    df_exploded = df_exploded.withColumn("polygon", F.explode("geo.coordinates"))
    df_exploded = df_exploded.withColumn("ring", F.explode("polygon"))
    df_exploded = df_exploded.withColumn("point", F.explode("ring"))
    df_exploded = df_exploded.withColumn("lon", F.col("point")[0])
    df_exploded = df_exploded.withColumn("lat", F.col("point")[1])

    df_bbox = (
        df_exploded.groupBy("location_id")
        .agg(
            ((F.min("lon") + F.max("lon")) / 2).alias("longitude"),
            ((F.min("lat") + F.max("lat")) / 2).alias("latitude")
        )
    )

    df = df.join(
        other=df_bbox,
        on="location_id",
        how="inner"
    ).select(
        df["*"],
        df_bbox["longitude"],
        df_bbox["latitude"]
    )

    schema_columns = [(field.name, field.dataType) for field in schema.fields]
    df = df.select([F.col(col_name).cast(col_dtype) for col_name, col_dtype in schema_columns])

    return df